# 05 — Download Data TradingView (tvkit)

> ⚠️ **Jalankan notebook ini di mesin LOKAL, bukan di Kaggle.**  
> Kaggle memblokir WebSocket ke TradingView.

**Tujuan:** Unduh 6 000 bar 15-menit untuk SPX, NIFTY, FTSE → simpan ke  
`data/raw/{symbol}_15m.parquet` sesuai format `src/data.py`.

**Keunggulan tvkit vs tvDatafeed:**
- Login Google OAuth didukung penuh (cookie browser, bukan password)
- Segmentasi otomatis untuk request > 5 000 bar
- API async modern dengan strong typing


## 1. Setup

In [ ]:
import sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    missing = []
    for pkg, imp in pkgs:
        try:
            __import__(imp)
        except ImportError:
            missing.append(pkg)
    if missing:
        print(f'Installing: {missing} ...')
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', '--user'] + missing,
            check=True,
        )
        print('Done.')

_ensure(('tvkit', 'tvkit'), ('browser-cookie3', 'browser_cookie3'))

REPO = Path().resolve()
while not (REPO / 'configs' / 'config.yaml').exists() and REPO != REPO.parent:
    REPO = REPO.parent
assert (REPO / 'configs' / 'config.yaml').exists(), f'config.yaml tidak ditemukan: {REPO}'
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import yaml, pandas as pd
import tvkit; print(f'tvkit {tvkit.__version__}')

cfg        = yaml.safe_load((REPO / 'configs' / 'config.yaml').read_text())
N_BARS     = cfg['data']['n_bars']
N_BARS_MIN = cfg['data']['n_bars_min']
INDICES    = cfg['data']['indices']

from src.paths import Paths
paths = Paths()
paths.raw.mkdir(parents=True, exist_ok=True)

print(f'Repo      : {REPO}')
print(f'Raw dir   : {paths.raw}')
print(f'N_BARS    : {N_BARS}  (min accepted: {N_BARS_MIN})')
print(f'Indices   : {[i["symbol"] for i in INDICES]}')


## 2. Autentikasi

tvkit membaca cookie `sessionid` dari profil Chrome/Firefox yang sudah login TradingView.  
**Tidak perlu password** — login Google OAuth bekerja karena cookie yang dipakai.

### Prasyarat
- Login ke TradingView di **Chrome** atau **Firefox** seperti biasa
- Windows: browser tidak harus ditutup; Linux: tutup dulu agar DB tidak terkunci

### Bila ekstraksi otomatis gagal
`F12` → **Application → Cookies → https://www.tradingview.com** → salin nilai `sessionid`


In [ ]:
import asyncio
from tvkit.api.chart.ohlcv import OHLCV
try:
    from tvkit.auth import BrowserCookieError
except ImportError:
    BrowserCookieError = Exception

AUTH_MODE   = None
AUTH_KWARGS = {}

for _browser in ('chrome', 'firefox'):
    try:
        async def _probe_browser(br):
            async with OHLCV(browser=br) as cl:
                return cl.account
        _acct = await _probe_browser(_browser)
        AUTH_MODE   = 'browser'
        AUTH_KWARGS = {'browser': _browser}
        print(f'✅ Browser auth OK via {_browser}')
        if _acct:
            print(f'   Akun    : {_acct}')
            print(f'   Max bars: {_acct.max_bars}')
        else:
            print('   (account info tidak tersedia)')
        break
    except Exception as e:
        print(f'⚠️  {_browser}: {type(e).__name__}: {e}')

if AUTH_MODE is None:
    print('\nEkstraksi otomatis gagal — masukkan sessionid manual:')
    print('  F12 → Application → Cookies → tradingview.com → sessionid')
    from getpass import getpass
    _sid = getpass('sessionid (tersembunyi): ').strip()
    if not _sid:
        raise RuntimeError('sessionid kosong. Login di Chrome/Firefox dulu.')
    AUTH_MODE   = 'cookies'
    AUTH_KWARGS = {'cookies': {'sessionid': _sid}}
    print('✅ Menggunakan sessionid manual.')

print(f'\nAUTH_MODE = {AUTH_MODE!r}')


## 3. Verifikasi Akun & Batas Bar


In [ ]:
async def check_caps():
    async with OHLCV(**AUTH_KWARGS) as cl:
        await cl.wait_until_ready()
        return cl.account

try:
    acct = await check_caps()
except Exception as e:
    # ProfileFetchError: TradingView mengubah frontend -> parser profil tvkit gagal.
    print(f'⚠️  Auth gagal ({type(e).__name__}) -> fallback mode ANONIM.')
    print('   Data tetap bisa diunduh, tetapi batas bar bisa lebih kecil.')
    AUTH_MODE, AUTH_KWARGS = 'anonymous', {}
    acct = None
if acct is None:
    print('⚠️  Info akun tidak tersedia — default 5 000 bar/segmen.')
    print(f'   N_BARS={N_BARS} di-handle lewat segmentasi otomatis.')
else:
    print(f'Akun     : {acct}')
    print(f'Max bars : {acct.max_bars}')
    sym_ok = '✅' if acct.max_bars >= N_BARS else 'ℹ️ '
    print(f'{sym_ok} max_bars={acct.max_bars}, N_BARS={N_BARS}')


## 4. Probe Exchange Prefix

tvkit memakai format `EXCHANGE:SYMBOL` (mis. `SP:SPX`).  
Cell ini mencoba beberapa kandidat dan melaporkan mana yang mengembalikan bar.


In [ ]:
CANDIDATES = {
    'SPX'  : ['SP:SPX',    'TVC:SPX',    'INDEX:SPX',  'NYSE:SPX'],
    'NIFTY': ['NSE:NIFTY', 'NSE:NIFTY50','TVC:NIFTY',  'INDEX:NIFTY'],
    'FTSE' : ['TVC:UKX',   'FTSE:UKX',   'LSE:UKX',    'TVC:FTSE'],
}
CONFIRMED = {}

async def probe_sym(exchange_symbol):
    try:
        async with OHLCV(**AUTH_KWARGS) as cl:
            bars = await cl.get_historical_ohlcv(
                exchange_symbol=exchange_symbol, interval='15', bars_count=5)
        return len(bars) if bars else None
    except Exception:
        return None

for sym, cands in CANDIDATES.items():
    print(f'\n--- {sym} ---')
    for cand in cands:
        n = await probe_sym(cand)
        flag = f'✅ {n} bar' if n else '❌'
        print(f'  {cand:25s}  {flag}')
        if n and sym not in CONFIRMED:
            CONFIRMED[sym] = cand

print('\n=== CONFIRMED ===')
for s, v in CONFIRMED.items():
    print(f'  {s:6s} -> {v}')
missing = [i['symbol'] for i in INDICES if i['symbol'] not in CONFIRMED]
if missing:
    print(f'\n⚠️  Belum terverifikasi: {missing}')
    print('   Tambahkan kandidat di dict CANDIDATES lalu jalankan ulang.')


## 5. Download Data

Download `N_BARS=6000` bar interval 15-menit per indeks.  
tvkit menangani segmentasi otomatis bila satu request tidak cukup.

**Format output (`src/data.py`):** DatetimeIndex UTC, kolom `[open,high,low,close,volume]`


In [ ]:
import time

def bars_to_df(bars) -> pd.DataFrame:
    rows = [{
        'dt'    : pd.Timestamp(b.timestamp, unit='s', tz='UTC'),
        'open'  : b.open,  'high': b.high,
        'low'   : b.low,   'close': b.close, 'volume': b.volume,
    } for b in bars]
    df = pd.DataFrame(rows).set_index('dt')
    df.index.name = None
    df = df.sort_index()
    return df[~df.index.duplicated(keep='first')]


# END_DATE = None -> ambil N_BARS bar TERBARU (sesuai metodologi paper).
# Data 15-menit TradingView hanya mundur ~5000 bar dari sekarang; tanggal lama
# (mis. Feb 2023/2024) menghasilkan respons kosong. Format bila diisi: 'YYYY-MM-DD'.
END_DATE   = None
LOOKBACK_D = 420   # hari kalender; cukup untuk 6000 bar (SPX ~230 hari bursa)
if END_DATE is not None:
    pd.Timestamp(END_DATE)   # gagal cepat bila tanggal tidak valid (mis. 2023-02-29)

async def dl_one(ex_sym, n_bars):
    async with OHLCV(**AUTH_KWARGS) as cl:
        if END_DATE is None:
            return await cl.get_historical_ohlcv(
                exchange_symbol=ex_sym, interval='15', bars_count=n_bars)
        start = (pd.Timestamp(END_DATE) - pd.Timedelta(days=LOOKBACK_D)).strftime('%Y-%m-%d')
        return await cl.get_historical_ohlcv(
            exchange_symbol=ex_sym, interval='15', start=start, end=END_DATE)


RESULTS = {}

for idx in INDICES:
    sym = idx['symbol']
    if sym not in CONFIRMED:
        print(f'\n⏭️  Lewati {sym} — jalankan Cell 4 dulu')
        continue
    ex_sym = CONFIRMED[sym]
    print(f'\n📥 {sym} ({ex_sym}) — {N_BARS} bar ...')
    bars = None
    for attempt in range(1, 4):
        try:
            bars = await dl_one(ex_sym, N_BARS)
            if bars: break
            print(f'   Attempt {attempt}: respons kosong')
        except Exception as e:
            print(f'   Attempt {attempt}: {type(e).__name__}: {e}')
        await asyncio.sleep(2 ** attempt)
    if not bars:
        print(f'❌ {sym}: gagal setelah 3 percobaan')
        continue
    df = bars_to_df(bars).tail(N_BARS)   # ambil N_BARS bar terakhir sebelum END_DATE
    RESULTS[sym] = df
    print(f'   ✅ {len(df)} bar  {df.index[0].date()} → {df.index[-1].date()}')

print(f'\nSelesai: {len(RESULTS)}/{len(INDICES)} indeks')


## 6. Validasi


In [ ]:
PASS = True
for sym, df in RESULTS.items():
    print(f'\n=== {sym} ===')
    n = len(df)
    bar_ok = n >= N_BARS_MIN
    print(f'  Bar count : {n:5d}  (min={N_BARS_MIN}) {"✅" if bar_ok else "❌"}')
    col_ok = list(df.columns) == ['open','high','low','close','volume']
    print(f'  Columns   : {list(df.columns)} {"✅" if col_ok else "❌"}')
    tz_ok = str(df.index.tz) == 'UTC'
    print(f'  Index tz  : {df.index.tz} {"✅" if tz_ok else "❌"}')
    nulls = df.isnull().sum().sum()
    print(f'  Nulls     : {nulls} {"✅" if nulls==0 else "⚠️"}')
    dups  = df.index.duplicated().sum()
    print(f'  Duplicates: {dups} {"✅" if dups==0 else "⚠️"}')
    mono = df.index.is_monotonic_increasing
    print(f'  Monotonic : {mono} {"✅" if mono else "❌"}')
    print(f'  Range     : {df.index[0]} → {df.index[-1]}')
    print(df.tail(3).to_string())
    ok = bar_ok and col_ok and tz_ok and mono
    if not ok:
        PASS = False
        print(f'  ❌ {sym} GAGAL validasi kritis!')
    else:
        print(f'  ✅ {sym} LULUS')
print()
if PASS:
    print('✅ Semua indeks lulus — lanjut Cell 7.')
else:
    print('❌ Ada kegagalan — periksa output sebelum menyimpan.')


## 7. Simpan Parquet

Format: `data/raw/{symbol}_15m.parquet` — sesuai kontrak `src/data.py`.


In [ ]:
for sym, df in RESULTS.items():
    out = paths.raw / f'{sym}_15m.parquet'
    if out.exists():
        ans = input(f'{out.name} sudah ada — timpa? [y/N]: ').strip().lower()
        if ans != 'y':
            print(f'  Lewati {sym}.')
            continue
    df.to_parquet(out)
    print(f'  ✅ {out.name}  ({out.stat().st_size/1024:.1f} KB  {len(df)} bar)')

print('\nFile di data/raw/:')
for f in sorted(paths.raw.glob('*.parquet')):
    df_tmp = pd.read_parquet(f)
    print(f'  {f.name:30s}  {f.stat().st_size/1024:7.1f} KB  '
          f'{len(df_tmp)} bar  {df_tmp.index[0].date()} → {df_tmp.index[-1].date()}')


## 8. Upload ke Kaggle sebagai Dataset

In [ ]:
print('=== Upload ke Kaggle ===')
print('Opsi A — via browser:')
print('  1. https://www.kaggle.com/datasets/new')
print(f'  2. Upload semua .parquet dari: {paths.raw}')
print('  3. Nama dataset: hblstm-index-15m-raw (atau pilihan Anda)')
print('  4. Kaggle Notebook → Add Data → file di /kaggle/input/<slug>/')
print()
print('Opsi B — Kaggle CLI:')
print(f'  cd "{paths.raw}"')
print( '  kaggle datasets init -p .')
print( '  # edit dataset-metadata.json')
print( '  kaggle datasets create -p .')


### Snippet untuk `00_run_all.ipynb`

Tambahkan cell ini **setelah bootstrap, sebelum `build_dataset()`**.  
Ganti `hblstm-index-15m-raw` dengan slug dataset Kaggle Anda.

```python
import shutil
from pathlib import Path

KAGGLE_DS = Path("/kaggle/input/hblstm-index-15m-raw")
if KAGGLE_DS.exists():
    for f in KAGGLE_DS.glob("*.parquet"):
        dest = paths.raw / f.name
        if not dest.exists():
            shutil.copy(f, dest)
            print(f"Copied : {f.name}")
        else:
            print(f"Cached : {f.name}")
else:
    print(f"⚠️  {KAGGLE_DS} tidak ditemukan — tambahkan via sidebar Kaggle → Add Data.")
```
